# Tutorial 11 — Detect data, domain, and concept shift

**Goal.** Compare two temporal windows, quantify numeric/categorical/prevalence drift, and combine the result with matured-label performance metrics.

**Prerequisites.** Base install; `poetry install -E ml` is optional. Runs offline on two bounded 1,000-payment windows.

**Produces.** A deterministic drift report, alert table, evidence-based interpretation, and a fingerprint for scheduled monitoring.


In [ ]:
import polars as pl

from fraudtwin.ml.drift import DriftConfig, compare_performance, compare_windows

reference = pl.DataFrame(
    {
        "amount": [10.0, 12.0, 15.0, 18.0, 20.0, 25.0],
        "channel": ["card"] * 5 + ["pix"],
        "label": [0, 0, 0, 1, 0, 1],
    }
)
comparison = pl.DataFrame(
    {
        "amount": [35.0, 40.0, 42.0, 50.0, 60.0, 70.0],
        "channel": ["pix"] * 5 + ["card"],
        "label": [0, 1, 1, 1, 1, 0],
    }
)
config = DriftConfig(
    reference_name="week-1",
    comparison_name="week-2",
    numeric_bins=4,
    psi_threshold=0.05,
    wasserstein_threshold=5.0,
    js_threshold=0.05,
)
report = compare_windows(reference, comparison, config)
print(report.reference_fingerprint, report.comparison_fingerprint)

## Read the evidence table

Data drift is a change in `P(X)`; domain shift is a change in customer, merchant, channel, geography, or scenario mix. A large alert is evidence to investigate, not proof that the model is broken.


In [ ]:
metrics = pl.DataFrame([metric.model_dump() for metric in report.metrics])
display(
    metrics.select(
        [
            "field",
            "field_type",
            "method",
            "reference_value",
            "comparison_value",
            "threshold",
            "alerted",
        ]
    )
)
print("alerts:", len(report.alerts))

## Separate concept and performance drift

Concept drift is a change in `P(Y|X)`. Compare the same metric and threshold policy on labels that have matured; unresolved labels are excluded and recorded in the report policy.


In [ ]:
performance = compare_performance(
    {"pr_auc": 0.42, "recall_at_fpr_1pct": 0.30},
    {"pr_auc": 0.31, "recall_at_fpr_1pct": 0.20},
    config,
    label_policy="matured_only",
)
display(pl.DataFrame([m.model_dump() for m in performance]))
print(
    {
        "label_policy": "matured_only",
        "interpretation": "performance change under a fixed evaluation policy",
    }
)

## Visualize and operationalize

Plot feature distributions and prevalence in a scheduled job. Alert thresholds should be calibrated against normal seasonal variation to avoid retraining on every payday or campaign launch.


In [ ]:
import json

assert report.fingerprint
assert report.manifest["config"]["minimum_samples"] == config.minimum_samples
print(
    json.dumps(
        {
            "report_version": report.report_version,
            "alerts": len(report.alerts),
            "fingerprint": report.fingerprint,
        },
        indent=2,
    )
)

Save `report.model_dump_json()` to your monitoring store and compare fingerprints across runs. Next: [Tutorial 12 — Kafka reliability and event-time correctness](12-avro-kafka-stream.ipynb) and the [drift and shift guide](../drift-and-shift.md).
